In [1]:
import pandas as pd
import numpy as np

from pycaret.clustering import (
    setup as cl_setup,
    create_model as cl_create_model,
    save_model as cl_save_model,
)

from pycaret.regression import (
    setup as reg_setup,
    save_model as reg_save_model,
    compare_models as reg_compare_models
)

In [3]:
# wczytanie danych z pliku csv

df = pd.read_csv("insurance.csv", delimiter=",")
df.columns.tolist()

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [4]:
# czyszczenie nazw kolumn
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns.tolist()

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [5]:
# sprawdzanie czy są brakujące wartości i ich usuwanie z DataFrame, kolumna po kolumnie

empty_values = df.isna().sum()

for key, value in empty_values.items():
    # print(f"{key}: {value}")
    if value > 0:
        df.dropna(subset=[key], inplace=True)

In [74]:
# cluster_data = df.drop(columns=['charges'])  # replace 'column_to_drop' with the actual column name you want to drop
# cluster_data.columns.tolist()

---
# **Klasteryzacja danych**

In [6]:
# Ustawienia klasteryzacji danych przy użyciu PyCaret

clustering_setup = cl_setup(
    data=df,
    session_id=123,
    normalize=True, # normalizacja danych przed klasteryzacją
    verbose=True, # wyświetlanie szczegółowych informacji podczas ustawiania klasteryzacji
    ignore_features=['charges'], # ignorowanie kolumny 'charges' podczas klasteryzacji
)

,Description,Value
0,Session id,123
1,Original data shape,"(1338, 7)"
2,Transformed data shape,"(1338, 9)"
3,Ignore features,1
4,Numeric features,3
5,Categorical features,3
6,Preprocess,True
7,Imputation type,simple
8,Numeric imputation,mean
9,Categorical imputation,mode


In [7]:
# stworzenie modelu k-means z 3 klastrami

model = cl_create_model('kmeans', num_clusters=3)


In [8]:
# zapis pipeline
cl_save_model(model, 'pipeline_clustering')
print()

Transformation Pipeline and Model Successfully Saved



---
# **Regresja**

In [9]:
# przygotowanie danych do analizy regresji
df_reg = df.copy()

# sprawdzenie typów wybranych kolumn i ewentualna konwersja do typów numerycznych
for column, dtype in dict(df_reg[['age', 'bmi', 'children', 'charges']].dtypes).items():
    if not np.issubdtype(dtype, np.number):
        df_reg[column] = pd.to_numeric(df_reg[column], errors='coerce')

    print(f"kolumna [{column}] jest teraz typu [{df_reg[column].dtype}]")

kolumna [age] jest teraz typu [int64]
kolumna [bmi] jest teraz typu [float64]
kolumna [children] jest teraz typu [int64]
kolumna [charges] jest teraz typu [float64]


In [10]:
regression_setup = reg_setup(
    data=df_reg,
    target='charges',
    session_id=123,
    normalize=True, # -> dane wejściowe są skalowane
    verbose=False # -> tryb cichy, minimalny output.
)

In [11]:
# porównanie modeli regresyjnych i wybór najlepszego

best_reg_model = reg_compare_models()
best_reg_model

GradientBoostingRegressor(random_state=123)

### Najlepszym modelem w naszym przypadku okazał się **Gradient Boosting Regressor**

In [12]:
# zapisanie modelu do pliku pipeline_regression
reg_save_model(best_reg_model, 'pipeline_regression')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['age', 'bmi', 'children'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['sex', 'smoker', 'region'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('ordinal_encoding',
                  TransformerWrapper(include=['sex', 'smoker'],
                                     transfor...
                                                                         {'col': 'smoker',
                                                                          'data_type': dtype('O'),
                                                                          'mapping': no     0
 yes    1
 NaN   -1
 dtype: int64}]))),
                 ('onehot_encoding',
                  TransformerWrapper(include=['region'],
    